In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS de_workspace26.shopeasy_raw_arnav;

In [0]:
%sql
use catalog de_workspace26;
use schema shopeasy_raw_arnav;

In [0]:
from pyspark.sql import functions as f;


In [0]:
df_orders = spark.read.csv(
    '/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/orders.csv',
    header=True, inferSchema=True
)

df_customers=spark.read.csv(
    '/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/customers.csv',
    header=True, inferSchema=True
)

df_products=spark.read.csv(
    '/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/products.csv',
    header=True, inferSchema=True
)


df_orders.show(5)

df_customers.show(5)
df_products.show(5)

In [0]:
df_orders = df_orders.withColumn("_ingested_at", f.current_timestamp()).withColumn("_source_file", f.lit("orders.csv"))
df_customers = df_customers.withColumn("_ingested_at", f.current_timestamp()).withColumn("_source_file", f.lit("customers.csv"))
df_products = df_products.withColumn("_ingested_at", f.current_timestamp()).withColumn("_source_file", f.lit("products.csv"))


df_orders.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable('bronze_orders')
df_customers.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable('bronze_customers')
df_products.write.format('delta').mode('overwrite').option('mergeSchema',True).saveAsTable('bronze_products')

In [0]:
%sql
ALTER TABLE bronze_orders ALTER COLUMN order_id SET NOT NULL;

In [0]:
%sql
describe history bronze_orders;

In [0]:
orders_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .option("pathGlobFilter", "orders*.csv")
  .option("cloudFiles.schemaLocation", 
          "/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/_schema/orders")
  .load("/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/"))

query_orders = (orders_stream.writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", 
          "/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/_checkpoints/orders_stream")
  .trigger(once=True)
  .toTable("de_workspace26.shopeasy_raw_arnav.orders_stream"))

query_orders.awaitTermination()
print("Orders done:", spark.read.table("de_workspace26.shopeasy_raw_arnav.orders_stream").count())

In [0]:
customers_stream = (spark.readStream
  .format("cloudFiles")
  .option("cloudFiles.format", "csv")
  .option("header", "true")
  .option("inferSchema", "true")
  .option("pathGlobFilter", "customers*.csv")
  .option("cloudFiles.schemaLocation", 
          "/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/_schema/customers")
  .load("/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/"))

query_customers = (customers_stream.writeStream
  .format("delta")
  .outputMode("append")
  .option("checkpointLocation", 
          "/Volumes/de_workspace26/shopeasy_raw_arnav/shopeasy_arnavg/_checkpoints/customers_stream")
  .trigger(once=True)
  .toTable("de_workspace26.shopeasy_raw_arnav.customers_stream"))

query_customers.awaitTermination()
print("Customers done:", spark.read.table("de_workspace26.shopeasy_raw_arnav.customers_stream").count())

In [0]:
orders=spark.read.table('bronze_orders')
orders.show(5)